# 02 · Ejecución: detección y localización 3D de tomates maduros y semimaduros

Este notebook contiene **todas las funciones** del sistema:

| Sección | Qué hace |
|---|---|
| 1. Calibración estéreo | Detecta el tablero, calibra el par de cámaras y rectifica las imágenes |
| 2. Profundidad (SGBM) | Calcula el mapa de disparidad entre la imagen izquierda y la derecha |
| 3. Detector | Segmenta los tomates **maduros y semimaduros** con YOLO11-seg |
| 4. Localización 3D | Combina máscara y disparidad para obtener X, Y, Z en metros |
| 5. Pipeline | `procesar_par()`: todo el proceso para un par de imágenes |
| 6. Visualización y evaluación | Gráficos, tabla y medición del error de Z |
| 7. Prueba sintética | Verifica el sistema sin cámaras |

Al final hay celdas de **demostración**. El notebook `03_api` carga este archivo con `DEMO = False`: así solo se definen las funciones y no se ejecutan las demos.

**Sistema de coordenadas:** cámara izquierda rectificada. X a la derecha, Y hacia abajo, Z hacia adelante (convención de OpenCV), en metros.

In [ ]:
# ---------------------------------------------------------------- configuración
import csv
import datetime as dt
import importlib.util
import math
import os
import subprocess
import sys
import threading
import time
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np

if "DEMO" not in globals():
    DEMO = True  # 03_api lo pone en False antes de cargar este notebook

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/tomates3d"   # carpeta del proyecto en Google Drive
    if importlib.util.find_spec("ultralytics") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
else:
    BASE_DIR = os.environ.get("TOMATES_DIR", str(Path.cwd().parent))  # carpeta del proyecto en la PC

WEIGHTS_PATH = os.path.join(BASE_DIR, "weights", "best.pt")   # generado por 01_entrenamiento
CALIB_PATH = os.path.join(BASE_DIR, "stereo_calib.yml")       # generado al calibrar desde la web
MIN_DEPTH_M = 0.25  # distancia mínima esperada: define el rango de búsqueda de disparidad

print(f"Colab: {EN_COLAB} | proyecto: {BASE_DIR}")
print(f"pesos: {WEIGHTS_PATH} ({'existe' if os.path.isfile(WEIGHTS_PATH) else 'NO existe'})")
print(f"calibración: {CALIB_PATH} ({'existe' if os.path.isfile(CALIB_PATH) else 'NO existe'})")

## 1. Calibración estéreo

Se fotografía un tablero de ajedrez de medidas conocidas con ambas cámaras a la vez, desde distintas posiciones. Con eso OpenCV calcula:
- los **intrínsecos** de cada cámara (focal, centro óptico, distorsión), con el método de Zhang;
- la **posición de una cámara respecto de la otra** (R, T), de donde sale la línea base *B*;
- las **transformaciones de rectificación**, que alinean ambas imágenes para que un mismo punto quede en la misma fila.

Los puntos 3D del tablero se generan en **metros**, así que todo lo métrico (línea base, X, Y, Z) sale en metros.

**Ajuste de escala con distancias conocidas.** La línea base es el parámetro que peor estima la calibración (con webcams no sincronizadas, un tablero no perfectamente plano o un tamaño de cuadro inexacto). Un error en *B* multiplica **todas** las distancias por el mismo factor, porque `Z = f·B/d`. Por eso se corrige midiendo el tomate a algunas distancias conocidas con cinta: `factor_escala()` calcula el *k* que mejor ajusta `k·Z_medida ≈ Z_real` y `escalar_calibracion()` multiplica la línea base por *k*. Es la misma idea que usan trabajos de localización de tomates con visión binocular (corrección por regresión contra distancias de referencia).

In [ ]:
SUBPIX_CRITERIA = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
STEREO_CRITERIA = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6)
MIN_PAIRS = 12


def find_corners(img: np.ndarray, pattern: tuple[int, int]) -> np.ndarray | None:
    """Busca las esquinas internas del tablero. Devuelve (N,1,2) float32 o None."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    # findChessboardCornersSB es más robusto a desenfoque e iluminación; si falla, se usa el clásico.
    if hasattr(cv2, "findChessboardCornersSB"):
        ok, corners = cv2.findChessboardCornersSB(
            gray, pattern, flags=cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_EXHAUSTIVE
        )
        if ok:
            return corners.astype(np.float32).reshape(-1, 1, 2)
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_FAST_CHECK
    ok, corners = cv2.findChessboardCorners(gray, pattern, flags=flags)
    if not ok:
        return None
    return cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), SUBPIX_CRITERIA)


def board_object_points(pattern: tuple[int, int], square_m: float) -> np.ndarray:
    cols, rows = pattern
    obj = np.zeros((rows * cols, 3), np.float32)
    obj[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2) * square_m
    return obj


@dataclass
class CalibPair:
    left: np.ndarray   # esquinas cámara izquierda (N,1,2)
    right: np.ndarray  # esquinas cámara derecha (N,1,2)


@dataclass
class CalibrationSession:
    """Acumula pares de imágenes del tablero enviados desde la web."""
    pattern: tuple[int, int]
    square_mm: float
    image_size: tuple[int, int] | None = None  # (w, h)
    pairs: dict[int, CalibPair] = field(default_factory=dict)
    _next_id: int = 0

    def add_pair(self, left: np.ndarray, right: np.ndarray) -> dict:
        h, w = left.shape[:2]
        if right.shape[:2] != (h, w):
            raise ValueError("Las imágenes izquierda y derecha tienen distinta resolución.")
        if self.image_size is None:
            self.image_size = (w, h)
        elif self.image_size != (w, h):
            raise ValueError(
                f"Resolución {w}x{h} distinta a la de la sesión {self.image_size[0]}x{self.image_size[1]}."
            )
        cl = find_corners(left, self.pattern)
        cr = find_corners(right, self.pattern)
        result = {"found_left": cl is not None, "found_right": cr is not None, "index": None}
        if cl is not None and cr is not None:
            idx = self._next_id
            self._next_id += 1
            self.pairs[idx] = CalibPair(cl, cr)
            result["index"] = idx
            # Región cubierta por el tablero (normalizada) para la guía de cobertura en la web.
            x, y, bw, bh = cv2.boundingRect(cl.reshape(-1, 2))
            result["coverage"] = [x / w, y / h, bw / w, bh / h]
        result["preview"] = _draw_preview(left, right, self.pattern, cl, cr)
        return result

    def remove_pair(self, idx: int) -> bool:
        return self.pairs.pop(idx, None) is not None

    def compute(self) -> tuple["StereoCalibration", dict]:
        if len(self.pairs) < MIN_PAIRS:
            raise ValueError(f"Se necesitan al menos {MIN_PAIRS} pares válidos (hay {len(self.pairs)}).")
        ids = sorted(self.pairs)
        obj = board_object_points(self.pattern, self.square_mm / 1000.0)
        objpts = [obj] * len(ids)
        ipl = [self.pairs[i].left for i in ids]
        ipr = [self.pairs[i].right for i in ids]
        size = self.image_size

        # K3 fijo: con webcams y pocos pares el 3er coeficiente radial suele sobreajustar.
        cflags = cv2.CALIB_FIX_K3
        rms_l, K1, D1, _, _ = cv2.calibrateCamera(objpts, ipl, size, None, None, flags=cflags)
        rms_r, K2, D2, _, _ = cv2.calibrateCamera(objpts, ipr, size, None, None, flags=cflags)
        res = cv2.stereoCalibrateExtended(
            objpts, ipl, ipr, K1, D1, K2, D2, size, None, None,
            flags=cv2.CALIB_FIX_INTRINSIC, criteria=STEREO_CRITERIA,
        )
        # OpenCV >= 4.9 agrega rvecs/tvecs al resultado; perViewErrors siempre es el último.
        rms, K1, D1, K2, D2, R, T = res[:7]
        per_view = res[-1]
        calib = StereoCalibration.from_stereo(K1, D1, K2, D2, R, T, size, rms)
        per_view = np.asarray(per_view).reshape(len(ids), -1).mean(axis=1)
        report = {
            "rms": float(rms),
            "rms_left": float(rms_l),
            "rms_right": float(rms_r),
            "baseline_cm": calib.baseline * 100,
            "focal_px": calib.f,
            "image_size": list(size),
            "n_pairs": len(ids),
            "per_pair": [{"index": i, "error": float(e)} for i, e in zip(ids, per_view)],
        }
        return calib, report


class StereoCalibration:
    """Parámetros estéreo + mapas de rectificación."""

    KEYS = ("K1", "D1", "K2", "D2", "R", "T", "R1", "R2", "P1", "P2", "Q")

    def __init__(self, image_size: tuple[int, int], rms: float, created: str, **mats: np.ndarray):
        self.image_size = tuple(int(v) for v in image_size)
        self.rms = float(rms)
        self.created = created
        for k in self.KEYS:
            setattr(self, k, np.asarray(mats[k], dtype=np.float64))
        w, h = self.image_size
        self._maps_l = cv2.initUndistortRectifyMap(self.K1, self.D1, self.R1, self.P1, (w, h), cv2.CV_32FC1)
        self._maps_r = cv2.initUndistortRectifyMap(self.K2, self.D2, self.R2, self.P2, (w, h), cv2.CV_32FC1)

    @classmethod
    def from_stereo(cls, K1, D1, K2, D2, R, T, image_size, rms) -> "StereoCalibration":
        # alpha=0: recorta a la región válida (sin bordes negros), mejor para la disparidad.
        R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(
            K1, D1, K2, D2, image_size, R, T, flags=cv2.CALIB_ZERO_DISPARITY, alpha=0
        )
        created = dt.datetime.now().isoformat(timespec="seconds")
        return cls(image_size, rms, created, K1=K1, D1=D1, K2=K2, D2=D2, R=R, T=T,
                   R1=R1, R2=R2, P1=P1, P2=P2, Q=Q)

    # --- parámetros del par rectificado ---
    @property
    def f(self) -> float:
        return float(self.P1[0, 0])

    @property
    def cx(self) -> float:
        return float(self.P1[0, 2])

    @property
    def cy(self) -> float:
        return float(self.P1[1, 2])

    @property
    def baseline(self) -> float:
        """Línea base en metros."""
        return float(abs(self.P2[0, 3] / self.P2[0, 0]))

    def rectify(self, left: np.ndarray, right: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        w, h = self.image_size
        if left.shape[1] != w or left.shape[0] != h or right.shape[:2] != left.shape[:2]:
            raise ValueError(
                f"Las imágenes deben ser {w}x{h} (resolución de calibración); "
                f"llegaron {left.shape[1]}x{left.shape[0]} y {right.shape[1]}x{right.shape[0]}."
            )
        rl = cv2.remap(left, *self._maps_l, cv2.INTER_LINEAR)
        rr = cv2.remap(right, *self._maps_r, cv2.INTER_LINEAR)
        return rl, rr

    def rectified_to_raw(self, pts: np.ndarray) -> np.ndarray:
        """Convierte píxeles del izquierdo rectificado a píxeles de la imagen izquierda original."""
        pts = np.asarray(pts, np.float64).reshape(-1, 2)
        if len(pts) == 0:
            return pts
        homog = np.hstack([pts, np.ones((len(pts), 1))])
        rays = (np.linalg.inv(self.P1[:, :3]) @ homog.T).T  # rayos en el sistema rectificado
        rays = (self.R1.T @ rays.T).T                       # al sistema de la cámara izquierda original
        raw, _ = cv2.projectPoints(rays.reshape(-1, 1, 3), np.zeros(3), np.zeros(3), self.K1, self.D1)
        return raw.reshape(-1, 2)

    def summary(self) -> dict:
        return {
            "image_size": list(self.image_size),
            "rms": self.rms,
            "baseline_cm": self.baseline * 100,
            "focal_px": self.f,
            "created": self.created,
        }

    # --- persistencia (yml de OpenCV) ---
    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fs = cv2.FileStorage(str(path), cv2.FILE_STORAGE_WRITE)
        fs.write("image_width", self.image_size[0])
        fs.write("image_height", self.image_size[1])
        fs.write("rms", self.rms)
        fs.write("created", self.created)
        for k in self.KEYS:
            fs.write(k, getattr(self, k))
        fs.release()

    @classmethod
    def load(cls, path: str | Path) -> "StereoCalibration":
        fs = cv2.FileStorage(str(path), cv2.FILE_STORAGE_READ)
        if not fs.isOpened():
            raise ValueError(f"No se pudo abrir {path}")
        try:
            size = (int(fs.getNode("image_width").real()), int(fs.getNode("image_height").real()))
            rms = fs.getNode("rms").real()
            created = fs.getNode("created").string() or ""
            mats = {k: fs.getNode(k).mat() for k in cls.KEYS[:6]}
        finally:
            fs.release()
        if any(m is None for m in mats.values()) or size[0] <= 0:
            raise ValueError("Archivo de calibración incompleto (faltan K1, D1, K2, D2, R, T o el tamaño).")
        # R1..Q se recalculan para aceptar también yml que solo traen intrínsecos + extrínsecos.
        calib = cls.from_stereo(mats["K1"], mats["D1"], mats["K2"], mats["D2"],
                                mats["R"], mats["T"], size, rms)
        calib.created = created or calib.created
        return calib


def calibracion_generica(ancho=1280, alto=720, fov_diagonal_deg=55.0, baseline_m=0.10) -> StereoCalibration:
    """Calibración TEÓRICA para dos cámaras iguales, paralelas y sin distorsión.

    Sirve para probar el sistema sin calibrar; la Z real puede tener errores de centímetros.
    Los valores por defecto corresponden a una Logitech C270 a 720p (FOV diagonal 55°).
    """
    f = (math.hypot(ancho, alto) / 2) / math.tan(math.radians(fov_diagonal_deg / 2))
    K = np.array([[f, 0, (ancho - 1) / 2], [0, f, (alto - 1) / 2], [0, 0, 1]])
    D = np.zeros((1, 5))
    T = np.array([[-baseline_m], [0.0], [0.0]])
    calib = StereoCalibration.from_stereo(K, D, K.copy(), D.copy(), np.eye(3), T, (ancho, alto), 0.0)
    calib.created = f"GENERICA (FOV diag {fov_diagonal_deg}°, {ancho}x{alto}, B={baseline_m * 100:.0f} cm)"
    return calib


def factor_escala(z_medidas: list[float], z_reales: list[float]) -> float:
    """Factor k que mejor ajusta k·Z_medida ≈ Z_real (mínimos cuadrados, recta por el origen).

    Un error en la línea base B multiplica todas las distancias por el mismo factor (Z = f·B/d), así
    que se corrige con un único parámetro con significado físico: B_corregida = k·B.
    """
    zm = np.asarray(z_medidas, float)
    zr = np.asarray(z_reales, float)
    if len(zm) == 0 or len(zm) != len(zr):
        raise ValueError("Se necesita al menos una medición (Z medida, Z real).")
    return float((zm * zr).sum() / (zm * zm).sum())


def escalar_calibracion(calib: "StereoCalibration", factor: float) -> "StereoCalibration":
    """Multiplica la línea base (T) por `factor`. La rectificación no cambia; X, Y y Z escalan igual."""
    if not 0.5 < factor < 1.5:
        raise ValueError(f"Factor de escala fuera de rango: {factor:.3f} (se espera cerca de 1).")
    nueva = StereoCalibration.from_stereo(calib.K1, calib.D1, calib.K2, calib.D2, calib.R, calib.T * factor,
                                          calib.image_size, calib.rms)
    nueva.created = f"{calib.created} | escala x{factor:.4f}"
    return nueva


def cargar_calibracion(ruta: str | None = None) -> StereoCalibration:
    ruta = ruta or CALIB_PATH
    if not os.path.isfile(ruta):
        raise FileNotFoundError(f"No existe {ruta}. Calibrá desde la web (03_api) o usá calibracion_generica().")
    return StereoCalibration.load(ruta)


def _draw_preview(left, right, pattern, cl, cr, max_w: int = 640) -> np.ndarray:
    views = []
    for img, corners in ((left, cl), (right, cr)):
        v = img.copy()
        if corners is not None:
            cv2.drawChessboardCorners(v, pattern, corners, True)
        else:
            cv2.putText(v, "TABLERO NO DETECTADO", (20, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        max(0.8, v.shape[1] / 900), (0, 0, 255), 3, cv2.LINE_AA)
        views.append(v)
    both = np.hstack(views)
    scale = max_w / both.shape[1]
    if scale < 1:
        both = cv2.resize(both, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return both


def epipolar_preview(rl: np.ndarray, rr: np.ndarray, step: int = 40) -> np.ndarray:
    """Par rectificado lado a lado con líneas horizontales: deben pasar por los mismos puntos."""
    both = np.hstack([rl, rr])
    for y in range(step // 2, both.shape[0], step):
        color = (0, 255, 0) if (y // step) % 2 == 0 else (0, 200, 255)
        cv2.line(both, (0, y), (both.shape[1] - 1, y), color, 1, cv2.LINE_AA)
    return both

## 2. Profundidad con SGBM

**Semi-Global Block Matching** (Hirschmüller, 2008), implementado en OpenCV. Para cada píxel del izquierdo rectificado busca, en la misma fila del derecho, el píxel más parecido. El desplazamiento es la **disparidad** *d*, y la profundidad vale `Z = f·B / d`.

Adaptaciones propias:
- **Relleno a la izquierda:** SGBM deja sin medir las primeras `numDisparities` columnas. Con líneas base grandes eso borraba casi toda la imagen.
- **Escala adaptativa:** se reduce la imagen para que el rango de búsqueda quede en unos 384 px. Es más rápido y la precisión en Z sigue siendo milimétrica.
- **Filtro WLS** (si está `opencv-contrib`): suaviza respetando bordes, pero solo donde SGBM midió algo.

In [ ]:
class SGBMMatcher:
    def __init__(self, num_disparities: int = 160, block_size: int = 5):
        num_disparities = int(np.ceil(num_disparities / 16) * 16)
        self.num_disparities = num_disparities
        self.left = cv2.StereoSGBM_create(
            minDisparity=0,
            numDisparities=num_disparities,
            blockSize=block_size,
            P1=8 * 3 * block_size ** 2,
            P2=32 * 3 * block_size ** 2,
            disp12MaxDiff=1,
            uniquenessRatio=10,
            speckleWindowSize=100,
            speckleRange=2,
            preFilterCap=63,
            mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY,
        )
        self.wls = None
        # En Colab conviven opencv-python y opencv-contrib-python: puede existir cv2.ximgproc sin el
        # filtro WLS. Si falta, se usa SGBM solo (funciona igual, con un poco más de ruido en bordes).
        xi = getattr(cv2, "ximgproc", None)
        if xi is not None and hasattr(xi, "createRightMatcher") and hasattr(xi, "createDisparityWLSFilter"):
            try:
                self.right = xi.createRightMatcher(self.left)
                self.wls = xi.createDisparityWLSFilter(self.left)
                self.wls.setLambda(8000.0)
                self.wls.setSigmaColor(1.5)
            except cv2.error:
                self.wls = None

    def __call__(self, rl: np.ndarray, rr: np.ndarray) -> np.ndarray:
        # Se rellena a la izquierda y luego se recorta, así cada columna se busca solo hasta
        # donde existe correspondencia (SGBM invalida las primeras numDisparities columnas).
        pad = self.num_disparities
        border = lambda img: cv2.copyMakeBorder(img, 0, 0, pad, 0, cv2.BORDER_CONSTANT, value=0)
        cl, cr = border(rl), border(rr)
        gl = cv2.cvtColor(cl, cv2.COLOR_BGR2GRAY)
        gr = cv2.cvtColor(cr, cv2.COLOR_BGR2GRAY)
        dl = self.left.compute(gl, gr)
        if self.wls is not None:
            dr = self.right.compute(gr, gl)
            disp = self.wls.filter(dl, cl, disparity_map_right=dr)
            # El WLS rellena huecos inventando valores: se conservan solo los píxeles que SGBM midió.
            disp[dl <= 0] = 0
        else:
            disp = dl
        disp = disp[:, pad:].astype(np.float32) / 16.0
        disp[disp <= 0] = np.nan
        return disp


_matchers: dict[int, SGBMMatcher] = {}
_matchers_lock = threading.Lock()


def get_matcher(num_disparities: int = 160) -> SGBMMatcher:
    with _matchers_lock:
        if num_disparities not in _matchers:
            _matchers[num_disparities] = SGBMMatcher(num_disparities)
        return _matchers[num_disparities]


def compute_disparity(rl: np.ndarray, rr: np.ndarray,
                      max_disparity: float = 160, max_width: int = 960, max_search: int = 384) -> np.ndarray:
    """Disparidad float32 en píxeles (NaN = inválida), alineada con la imagen izquierda rectificada.

    max_disparity: disparidad máxima esperada a resolución completa (= f·B / Z_mín).
    """
    h, w = rl.shape[:2]
    scale = min(1.0, max_width / w, max(0.4, max_search / max(max_disparity, 1)))
    if scale < 1:
        size = (int(round(w * scale)), int(round(h * scale)))
        rl_s = cv2.resize(rl, size, interpolation=cv2.INTER_AREA)
        rr_s = cv2.resize(rr, size, interpolation=cv2.INTER_AREA)
    else:
        rl_s, rr_s = rl, rr
    # Con líneas base grandes la disparidad cercana supera la mitad del ancho: se permite casi todo.
    nd = int(np.ceil(min(max_disparity * scale, rl_s.shape[1] - 64) / 16) * 16)
    disp = get_matcher(max(16, nd))(rl_s, rr_s)
    if scale < 1:
        disp = cv2.resize(disp, (w, h), interpolation=cv2.INTER_NEAREST) / scale
    return disp


def disparidad_en_franjas(rl: np.ndarray, rr: np.ndarray, cajas: list, max_disparity: float) -> np.ndarray:
    """Disparidad solo en las franjas horizontales que contienen tomates (el resto queda NaN).

    En un par rectificado cada punto tiene su correspondencia en la misma fila, así que basta con
    procesar las filas de cada caja (más un margen) a todo lo ancho. Es varias veces más rápido que
    la imagen completa y da el mismo resultado sobre los tomates.
    """
    h = rl.shape[0]
    franjas = []
    for x1, y1, x2, y2 in cajas:
        margen = max(16, int(0.2 * (y2 - y1)))
        franjas.append([max(0, int(y1) - margen), min(h, int(y2) + margen + 1)])
    franjas.sort()
    unidas = []
    for a, b in franjas:
        if unidas and a <= unidas[-1][1]:
            unidas[-1][1] = max(unidas[-1][1], b)
        else:
            unidas.append([a, b])
    disp = np.full(rl.shape[:2], np.nan, np.float32)
    for a, b in unidas:
        disp[a:b] = compute_disparity(rl[a:b], rr[a:b], max_disparity=max_disparity)
    return disp


def colorize_depth(disp: np.ndarray, f: float, baseline: float, z_max: float = 2.0) -> np.ndarray:
    """Mapa de profundidad coloreado (cerca = cálido) para mostrar."""
    with np.errstate(divide="ignore", invalid="ignore"):
        z = f * baseline / disp
    valid = np.isfinite(z) & (z > 0) & (z < z_max)
    norm = np.zeros(disp.shape, np.uint8)
    norm[valid] = (255 * (1 - z[valid] / z_max)).astype(np.uint8)
    color = cv2.applyColorMap(norm, cv2.COLORMAP_TURBO)
    color[~valid] = 0
    return color


def depth_at(disp: np.ndarray, x: float, y: float, f: float, baseline: float, win: int = 7):
    """Profundidad (m) en un punto del izquierdo rectificado: mediana de una ventana."""
    h, w = disp.shape
    x, y = int(round(x)), int(round(y))
    r = win // 2
    patch = disp[max(0, y - r):min(h, y + r + 1), max(0, x - r):min(w, x + r + 1)]
    vals = patch[np.isfinite(patch) & (patch > 0)]
    if vals.size == 0:
        return None, None
    d = float(np.median(vals))
    return f * baseline / d, d

## 3. Detector de tomates (YOLO11-seg)

El modelo entrenado en `01_entrenamiento` tiene **una sola clase**, `tomate`, que reúne maduros y semimaduros (los verdes se ignoran). Devuelve, por cada tomate, la confianza, la caja y la **máscara** (los píxeles del fruto). La máscara permite medir la profundidad solo sobre el tomate, sin hojas ni fondo.

Si todavía no existe `best.pt`, se usa el modelo COCO `yolo11n-seg.pt` y se toman como "tomate" las clases *apple*, *orange* y *sports ball*. Sirve para probar el sistema con una pelota o una naranja.

In [ ]:
PROXY_CLASSES = {"apple", "orange", "sports ball"}


@dataclass
class Detection:
    conf: float
    bbox: tuple[float, float, float, float]  # x1, y1, x2, y2
    mask: np.ndarray                          # bool HxW, mismo tamaño que la imagen


class TomatoDetector:
    def __init__(self, weights: str | None = None, imgsz: int = 640):
        from ultralytics import YOLO

        weights = weights or WEIGHTS_PATH
        self.proxy = not Path(weights).is_file()
        if self.proxy:
            print(f"[detector] No se encontró '{weights}'. Usando yolo11n-seg.pt (COCO) con clases sustitutas "
                  f"{sorted(PROXY_CLASSES)}.")
            weights = "yolo11n-seg.pt"
        self.weights = str(weights)
        self.model = YOLO(weights)
        if self.model.task != "segment":
            raise ValueError("Se necesita un modelo de segmentación (yolo11*-seg).")
        self.imgsz = imgsz
        import torch
        self.gpu = torch.cuda.is_available()  # en GPU se usa media precisión (FP16): ~2x más rápido
        self._lock = threading.Lock()

    def info(self) -> dict:
        return {"weights": self.weights, "proxy": self.proxy, "gpu": self.gpu}

    def __call__(self, img: np.ndarray, conf: float = 0.35) -> list[Detection]:
        with self._lock:
            res = self.model.predict(img, imgsz=self.imgsz, conf=conf, retina_masks=True, verbose=False,
                                     device=0 if self.gpu else "cpu", half=self.gpu)[0]
        h, w = img.shape[:2]
        if res.boxes is None or res.masks is None:
            return []
        boxes = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        clss = res.boxes.cls.cpu().numpy().astype(int)
        masks = res.masks.data.cpu().numpy()
        out = []
        for box, c, k, m in zip(boxes, confs, clss, masks):
            if self.proxy and self.model.names[k] not in PROXY_CLASSES:
                continue
            if m.shape != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            out.append(Detection(float(c), tuple(float(v) for v in box), m > 0.5))
        return out

## 4. Localización 3D

Para cada tomate:
1. Se **erosiona** la máscara para no medir en el borde, donde la disparidad mezcla fruto y fondo.
2. Se toma la **mediana** de la disparidad entre los percentiles 20 y 80 (descarta valores atípicos).
3. Con la disparidad *d*, la focal *f*, la línea base *B* y el punto principal (cx, cy) de la calibración:
   `Z = f·B/d`,  `X = (u − cx)·Z/f`,  `Y = (v − cy)·Z/f`, donde (u, v) es el centroide de la máscara.
4. **Diámetro** ≈ `2·√(área/π)·Z/f`. Como la estéreo mide la superficie visible, el **centro** del fruto se estima sumando un radio a Z.

**Segundo método: máscaras en ambas cámaras.** La piel del tomate es lisa, roja y con brillos: tiene poca textura, así que SGBM encuentra pocos píxeles válidos (`depth_valid_ratio` bajo) y Z se vuelve ruidosa. Por eso también se detecta el tomate en la imagen **derecha** y se lo empareja con el de la izquierda (misma fila, área parecida, disparidad dentro del rango). La disparidad es la diferencia horizontal entre los **centroides de las dos máscaras**, `d = u_izq − u_der`. Para un fruto aproximadamente esférico, el centro de la silueta es la proyección de su centro, así que esto da **directamente el centro** del fruto sin depender de la textura.

`xyz_center_m` usa el método de máscaras cuando hay pareja y SGBM si no la hay (por ejemplo, si el tomate está tapado de forma distinta en cada cámara o toca el borde). El campo `metodo_z` indica cuál se usó; ambos resultados quedan en la respuesta para compararlos.

In [ ]:
MIN_VALID_PIXELS = 15
COLOR_TOMATE = (40, 40, 230)  # BGR


def robust_disparity(disp: np.ndarray, mask: np.ndarray) -> tuple[float | None, float]:
    """Mediana de la disparidad dentro de la máscara erosionada. Devuelve (disparidad, fracción válida)."""
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None, 0.0
    size = max(xs.max() - xs.min(), ys.max() - ys.min())
    k = max(3, int(size * 0.15) | 1)
    core = cv2.erode(mask.astype(np.uint8), np.ones((k, k), np.uint8)).astype(bool)
    if core.sum() < MIN_VALID_PIXELS:
        core = mask
    vals = disp[core]
    vals = vals[np.isfinite(vals) & (vals > 0)]
    valid_ratio = len(vals) / max(1, core.sum())
    if len(vals) < MIN_VALID_PIXELS:
        return None, valid_ratio
    lo, hi = np.percentile(vals, [20, 80])
    vals = vals[(vals >= lo) & (vals <= hi)]
    return float(np.median(vals)), valid_ratio


def mask_polygon(mask: np.ndarray, max_points: int = 40) -> list[list[float]]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    c = max(contours, key=cv2.contourArea)
    eps = 0.005 * cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, eps, True)
    while len(approx) > max_points:
        eps *= 1.5
        approx = cv2.approxPolyDP(c, eps, True)
    return [[float(x), float(y)] for x, y in approx.reshape(-1, 2)]


def _centroide(mask: np.ndarray):
    m = cv2.moments(mask.astype(np.uint8), binaryImage=True)
    if m["m00"] == 0:
        return None
    return m["m10"] / m["m00"], m["m01"] / m["m00"], float(m["m00"])


def _toca_borde(bbox, w: int, margen: int = 3) -> bool:
    return bbox[0] <= margen or bbox[2] >= w - 1 - margen


def emparejar_mascaras(dets_izq: list[Detection], dets_der: list[Detection], ancho: int,
                       max_disparity: float) -> dict[int, tuple[float, float, float]]:
    """Empareja cada tomate de la izquierda con el mismo tomate en la derecha (imágenes rectificadas).

    Criterios: misma fila (el par está rectificado), disparidad positiva y dentro del rango, área
    parecida y que ninguna de las dos máscaras toque el borde lateral (centroide sesgado).
    Devuelve {índice izquierdo: (u, v, área) del tomate derecho}.
    """
    der = [(j, _centroide(d.mask), d) for j, d in enumerate(dets_der)]
    der = [(j, c, d) for j, c, d in der if c is not None and not _toca_borde(d.bbox, ancho)]
    candidatos = []
    for i, d in enumerate(dets_izq):
        ci = _centroide(d.mask)
        if ci is None or _toca_borde(d.bbox, ancho):
            continue
        ui, vi, ai = ci
        alto = d.bbox[3] - d.bbox[1]
        for j, (uj, vj, aj), _ in der:
            disp = ui - uj
            dy = abs(vi - vj)
            ratio = aj / ai
            if 0.5 < disp <= 1.2 * max_disparity and dy <= max(12.0, 0.25 * alto) and 0.6 <= ratio <= 1.6:
                candidatos.append((dy / max(alto, 1) + abs(1 - ratio), i, j, (uj, vj, aj)))
    pares, usados_i, usados_j = {}, set(), set()
    for _, i, j, cj in sorted(candidatos):  # el par más parecido primero
        if i not in usados_i and j not in usados_j:
            pares[i] = cj
            usados_i.add(i)
            usados_j.add(j)
    return pares


def localize(dets: list[Detection], disp: np.ndarray, calib: StereoCalibration,
             pares: dict | None = None) -> list[dict]:
    """Calcula X, Y, Z de cada tomate con dos métodos:

    - SGBM: mediana de la disparidad dentro de la máscara → superficie visible (+1 radio = centro).
    - Máscaras: diferencia horizontal entre el centroide de la máscara izquierda y el de la derecha →
      directamente el centro del fruto (para un fruto ~esférico, el centro de la silueta es la
      proyección del centro). No depende de la textura de la piel.
    `xyz_center_m` usa máscaras si hay pareja y SGBM si no (campo `metodo_z`).
    """
    f, B, cx, cy = calib.f, calib.baseline, calib.cx, calib.cy
    pares = pares or {}
    results = []
    for i, d in enumerate(dets):
        c = _centroide(d.mask)
        if c is None:
            continue
        u, v, area = c
        polygon = mask_polygon(d.mask)
        item = {
            "id": i,
            "conf": round(d.conf, 3),
            "bbox": [round(float(c), 1) for c in d.bbox],
            "centroid_px": [round(u, 1), round(v, 1)],
            "polygon": polygon,
            "disparity_px": None,         # SGBM
            "xyz_m": None,                # SGBM: superficie visible
            "xyz_center_sgbm_m": None,    # SGBM: superficie + 1 radio
            "disparity_mask_px": None,    # máscaras
            "xyz_mask_m": None,           # máscaras: centro del fruto
            "xyz_center_m": None,         # el preferido de los dos
            "metodo_z": None,
            "diameter_m": None,
            "depth_valid_ratio": 0.0,
        }
        radio_px = np.sqrt(area / np.pi)
        disp_val, ratio = robust_disparity(disp, d.mask)
        item["depth_valid_ratio"] = round(ratio, 2)
        if disp_val is not None:
            Z = f * B / disp_val
            X = (u - cx) * Z / f
            Y = (v - cy) * Z / f
            Zc = Z + radio_px * Z / f
            item.update({
                "disparity_px": round(disp_val, 2),
                "xyz_m": [round(X, 4), round(Y, 4), round(Z, 4)],
                "xyz_center_sgbm_m": [round(X * Zc / Z, 4), round(Y * Zc / Z, 4), round(Zc, 4)],
            })
        if i in pares:
            ud, _, _ = pares[i]
            dm = u - ud
            Zm = f * B / dm
            item.update({
                "disparity_mask_px": round(dm, 2),
                "xyz_mask_m": [round((u - cx) * Zm / f, 4), round((v - cy) * Zm / f, 4), round(Zm, 4)],
            })
        if item["xyz_mask_m"]:
            item["xyz_center_m"], item["metodo_z"] = item["xyz_mask_m"], "mascaras"
        elif item["xyz_center_sgbm_m"]:
            item["xyz_center_m"], item["metodo_z"] = item["xyz_center_sgbm_m"], "sgbm"
        if item["xyz_center_m"]:
            item["diameter_m"] = round(float(2 * radio_px * item["xyz_center_m"][2] / f), 4)
        # Coordenadas en la imagen izquierda ORIGINAL (sin rectificar), para el overlay en vivo.
        raw = calib.rectified_to_raw(np.array([[u, v]] + polygon, np.float64))
        item["centroid_raw_px"] = [round(float(raw[0, 0]), 1), round(float(raw[0, 1]), 1)]
        item["polygon_raw"] = [[round(float(x), 1), round(float(y), 1)] for x, y in raw[1:]]
        results.append(item)
    return results


def draw(img: np.ndarray, results: list[dict]) -> np.ndarray:
    out = img.copy()
    overlay = img.copy()
    for r in results:
        pts = np.array(r["polygon"], np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(overlay, [pts], COLOR_TOMATE)
            cv2.polylines(out, [pts], True, COLOR_TOMATE, 2, cv2.LINE_AA)
    out = cv2.addWeighted(overlay, 0.35, out, 0.65, 0)
    scale = max(0.5, img.shape[1] / 1280)
    for r in results:
        u, v = (int(c) for c in r["centroid_px"])
        cv2.drawMarker(out, (u, v), (255, 255, 255), cv2.MARKER_CROSS, 14, 2)
        z = r["xyz_center_m"][2] if r["xyz_center_m"] else None
        label = f"#{r['id']} " + (f"Z={z:.2f}m" if z else "Z=?")
        x1, y1 = int(r["bbox"][0]), int(r["bbox"][1])
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        cv2.rectangle(out, (x1, y1 - th - 8), (x1 + tw + 6, y1), (20, 20, 20), -1)
        cv2.putText(out, label, (x1 + 3, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    return out

## 5. Pipeline completo

`procesar_par()` recibe las imágenes izquierda y derecha **originales** (BGR, como las lee `cv2.imread`) y:
1. las rectifica con la calibración;
2. detecta los tomates (maduros y semimaduros) en **ambas** imágenes rectificadas y los empareja;
3. calcula la disparidad con SGBM (en vivo, solo en las franjas de los tomates);
4. calcula X, Y, Z de cada tomate con los dos métodos.

Es la misma función que usa la API del notebook 03.

In [ ]:
def procesar_par(left: np.ndarray, right: np.ndarray, calib: StereoCalibration, detector,
                 conf: float = 0.35, imagenes: bool = True, min_depth_m: float | None = None) -> dict:
    min_depth_m = min_depth_m or MIN_DEPTH_M
    t0 = time.perf_counter()
    rl, rr = calib.rectify(left, right)  # ValueError si la resolución no coincide con la calibración
    t1 = time.perf_counter()
    dets = detector(rl, conf=conf)
    dets_der = detector(rr, conf=conf) if dets else []
    t2 = time.perf_counter()
    max_disp = calib.f * calib.baseline / min_depth_m
    pares = emparejar_mascaras(dets, dets_der, rl.shape[1], max_disp)
    if imagenes:  # el mapa de profundidad completo solo hace falta para mostrarlo
        disp = compute_disparity(rl, rr, max_disparity=max_disp)
    else:         # modo en vivo: solo las filas de los tomates detectados
        disp = disparidad_en_franjas(rl, rr, [d.bbox for d in dets], max_disp)
    t3 = time.perf_counter()
    tomates = localize(dets, disp, calib, pares)
    t4 = time.perf_counter()
    out = {
        "tomatoes": tomates,
        "count": len(tomates),
        "image_size": list(calib.image_size),
        "camera": {"f_px": calib.f, "cx": calib.cx, "cy": calib.cy, "baseline_m": calib.baseline},
        "timing_ms": {
            "rectify": round((t1 - t0) * 1000, 1),
            "detect": round((t2 - t1) * 1000, 1),
            "disparity": round((t3 - t2) * 1000, 1),
            "localize": round((t4 - t3) * 1000, 1),
            "total": round((t4 - t0) * 1000, 1),
        },
        "proxy_model": bool(getattr(detector, "proxy", False)),
    }
    if imagenes:
        out["annotated"] = draw(rl, tomates)
        out["depth"] = colorize_depth(disp, calib.f, calib.baseline)
    return out

## 6. Visualización y evaluación

- `mostrar_resultado()`: imagen anotada, mapa de profundidad, tabla de coordenadas y gráfico 3D.
- `evaluar_profundidad()`: compara la Z medida con distancias tomadas con cinta métrica. Usa un CSV con columnas `left,right,z_true_m` (una fila por par, con un tomate en escena) y reporta MAE, RMSE y sesgo.

In [ ]:
def leer_imagen(ruta: str) -> np.ndarray:
    img = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"No se pudo leer la imagen {ruta}")
    return img


def tabla_resultado(res: dict):
    import pandas as pd
    filas = []
    for t in res["tomatoes"]:
        p = t["xyz_center_m"] or [None] * 3
        filas.append({
            "id": t["id"], "conf": t["conf"],
            "X (m)": p[0], "Y (m)": p[1],
            "Z centro máscaras (m)": t["xyz_mask_m"][2] if t["xyz_mask_m"] else None,
            "Z centro SGBM (m)": t["xyz_center_sgbm_m"][2] if t["xyz_center_sgbm_m"] else None,
            "Z superficie SGBM (m)": t["xyz_m"][2] if t["xyz_m"] else None,
            "método": t["metodo_z"],
            "Ø (cm)": t["diameter_m"] * 100 if t["diameter_m"] else None,
            "disp. válida": t["depth_valid_ratio"],
        })
    return pd.DataFrame(filas)


def mostrar_resultado(res: dict):
    import matplotlib.pyplot as plt
    from IPython.display import display

    fig = plt.figure(figsize=(16, 5))
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(cv2.cvtColor(res["annotated"], cv2.COLOR_BGR2RGB))
    ax1.set_title(f"{res['count']} tomate(s)")
    ax1.axis("off")
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(cv2.cvtColor(res["depth"], cv2.COLOR_BGR2RGB))
    ax2.set_title("Profundidad (cerca = cálido)")
    ax2.axis("off")
    ax3 = fig.add_subplot(1, 3, 3, projection="3d")
    pts = [t["xyz_center_m"] for t in res["tomatoes"] if t["xyz_center_m"]]
    ax3.scatter([0], [0], [0], c="k", marker="^", s=60, label="cámara")
    if pts:
        pts = np.array(pts)
        # Ejes del gráfico: X derecha, Z profundidad, -Y arriba.
        ax3.scatter(pts[:, 0], pts[:, 2], -pts[:, 1], c="red", s=120, label="tomates")
        for t in res["tomatoes"]:
            if t["xyz_center_m"]:
                x, y, z = t["xyz_center_m"]
                ax3.text(x, z, -y, f" #{t['id']}")
    ax3.set_xlabel("X (m)")
    ax3.set_ylabel("Z (m)")
    ax3.set_zlabel("-Y (m)")
    ax3.legend()
    plt.tight_layout()
    plt.show()
    display(tabla_resultado(res))
    print("tiempos (ms):", res["timing_ms"])


def evaluar_profundidad(csv_path: str, calib: StereoCalibration, detector, campo: str = "xyz_center_m") -> dict:
    """campo: "xyz_center_m" (preferido), "xyz_mask_m", "xyz_center_sgbm_m" o "xyz_m" (superficie SGBM)."""
    key = campo
    base = Path(csv_path).parent
    errores = []
    with open(csv_path, newline="") as fh:
        for row in csv.DictReader(fh):
            res = procesar_par(leer_imagen(base / row["left"]), leer_imagen(base / row["right"]),
                               calib, detector, imagenes=False)
            found = [t for t in res["tomatoes"] if t[key]]
            z_true = float(row["z_true_m"])
            if not found:
                print(f"{row['left']}: sin detección con profundidad válida")
                continue
            z = max(found, key=lambda t: t["conf"])[key][2]
            errores.append(z - z_true)
            print(f"{row['left']}: Z={z:.3f} m  real={z_true:.3f} m  error={100 * (z - z_true):+.1f} cm")
    if not errores:
        return {}
    n = len(errores)
    resumen = {
        "n": n,
        "mae_cm": 100 * sum(abs(e) for e in errores) / n,
        "rmse_cm": 100 * math.sqrt(sum(e * e for e in errores) / n),
        "sesgo_cm": 100 * sum(errores) / n,
    }
    print(f"\nn={n}  MAE={resumen['mae_cm']:.2f} cm  RMSE={resumen['rmse_cm']:.2f} cm  sesgo={resumen['sesgo_cm']:+.2f} cm")
    return resumen

## 7. Prueba sintética (sin cámaras)

Simula un par estéreo con valores conocidos para verificar todo el sistema sin hardware:
1. **Calibración:** renderiza un tablero en 22 posiciones vistas por dos cámaras separadas 8 cm y comprueba que la línea base calculada sea ≈ 8 cm.
2. **Profundidad:** renderiza un "tomate" (disco texturado) a 30, 45 y 60 cm con la calibración genérica, y compara la Z medida con la real. Como no hay red entrenada, el detector se reemplaza por uno que devuelve la máscara conocida.

In [ ]:
class DetectorConocido:
    """Detector de prueba: devuelve una máscara fija (la del tomate simulado)."""
    proxy = False

    def __init__(self, mask: np.ndarray, mask_der: np.ndarray | None = None):
        # procesar_par llama primero con la imagen izquierda y después con la derecha.
        self.masks = [mask, mask_der if mask_der is not None else mask]
        self._n = 0

    def info(self):
        return {"weights": "prueba", "proxy": False}

    def __call__(self, img, conf=0.35):
        mask = self.masks[self._n % 2]
        self._n += 1
        ys, xs = np.nonzero(mask)
        return [Detection(0.99, (xs.min(), ys.min(), xs.max(), ys.max()), mask)]


def _proyectar_plano(K, tex, Rc, tc, Rw, tw, S, size):
    """Proyecta una textura plana (pose Rw, tw en la cámara izquierda) en una cámara (Rc, tc)."""
    R = Rc @ Rw
    t = Rc @ tw + tc
    Hm = K @ np.column_stack([R[:, 0], R[:, 1], t]) @ S
    img = cv2.warpPerspective(tex, Hm, size)
    mask = cv2.warpPerspective(np.full(tex.shape[:2], 255, np.uint8), Hm, size) > 0
    return img, mask, Hm


def prueba_calibracion_sintetica(baseline_m=0.08, n_pares=22, seed=0) -> dict:
    rng = np.random.default_rng(seed)
    W, H = 1280, 720
    K = np.array([[900, 0, 640], [0, 900, 360], [0, 0, 1]], float)
    R_lr = cv2.Rodrigues(np.array([0.0, np.deg2rad(1.5), 0.0]))[0]
    T_lr = np.array([-baseline_m, 0.002, 0.001])
    cols, rows, sq, px = 9, 6, 0.025, 60
    board = np.full(((rows + 3) * px, (cols + 3) * px), 255, np.uint8)
    for r in range(rows + 1):
        for c in range(cols + 1):
            if (r + c) % 2 == 0:
                board[(r + 1) * px:(r + 2) * px, (c + 1) * px:(c + 2) * px] = 0
    S = np.array([[sq / px, 0, -2 * sq], [0, sq / px, -2 * sq], [0, 0, 1]])

    def render(Rw, tw, Rc, tc):
        img, mask, _ = _proyectar_plano(K, board, Rc, tc, Rw, tw, S, (W, H))
        bg = rng.integers(90, 160, (H, W), dtype=np.uint8)
        out = cv2.GaussianBlur(np.where(mask, img, bg), (3, 3), 0)
        return cv2.cvtColor(out, cv2.COLOR_GRAY2BGR)

    sesion = CalibrationSession((cols, rows), sq * 1000)
    for _ in range(n_pares):
        Rw = cv2.Rodrigues(rng.uniform(-0.45, 0.45, 3) + np.array([np.pi, 0, 0]))[0]
        tw = np.array([rng.uniform(-0.15, 0.05), rng.uniform(-0.08, 0.02), rng.uniform(0.45, 0.9)])
        sesion.add_pair(render(Rw, tw, np.eye(3), np.zeros(3)), render(Rw, tw, R_lr, T_lr))
    _, rep = sesion.compute()
    print(f"Calibración sintética: {rep['n_pairs']} pares, RMS={rep['rms']:.3f} px, "
          f"línea base={rep['baseline_cm']:.2f} cm (real {baseline_m * 100:.2f} cm)")
    return rep


def escena_tomate(calib: StereoCalibration, Z: float, X: float, Y: float, radio=0.035, seed=1):
    """Par estéreo sintético: fondo plano lejano + disco texturado (tomate) en (X, Y, Z)."""
    rng = np.random.default_rng(seed)
    W, H = calib.image_size
    K, B = calib.K1, calib.baseline
    tex = cv2.GaussianBlur(cv2.cvtColor(rng.integers(0, 255, (400, 400), dtype=np.uint8), cv2.COLOR_GRAY2BGR), (5, 5), 0)
    disco = np.zeros(tex.shape[:2], np.uint8)
    cv2.circle(disco, (200, 200), 199, 255, -1)
    imgs = []
    for tc in (np.zeros(3), np.array([-B, 0, 0])):
        def plano(z, size_m, cx_m, cy_m, shift=0):
            s = size_m / tex.shape[1]
            S = np.array([[s, 0, cx_m - size_m / 2], [0, s, cy_m - size_m / 2], [0, 0, 1]])
            t = np.clip(tex.astype(int) + shift, 0, 255).astype(np.uint8)
            return _proyectar_plano(K, t, np.eye(3), tc, np.eye(3), np.array([0, 0, z]), S, (W, H))
        fondo, _, _ = plano(Z + 0.4, 3.0, B / 2, 0)
        fruto, _, Hm = plano(Z, 2 * radio, X, Y, 40)
        m = cv2.warpPerspective(disco, Hm, (W, H)) > 0
        out = fondo.copy()
        out[m] = fruto[m]
        imgs.append((out, m))
    (L, mL), (R, mR) = imgs
    return L, R, mL, mR


def prueba_profundidad_sintetica(calib: StereoCalibration, distancias=(0.30, 0.45, 0.60)) -> list[dict]:
    filas = []
    for Z in distancias:
        X, Y = calib.baseline / 2, 0.02  # centrado en la zona que ven ambas cámaras
        L, R, mL, mR = escena_tomate(calib, Z, X, Y)
        a3 = lambda m: np.dstack([m.astype(np.uint8) * 255] * 3)
        mask_l, mask_r = (x[:, :, 0] > 127 for x in calib.rectify(a3(mL), a3(mR)))
        res = procesar_par(L, R, calib, DetectorConocido(mask_l, mask_r), imagenes=False)
        t = res["tomatoes"][0]
        # El "tomate" sintético es un disco plano: su superficie y su centro están a la misma Z.
        zs = t["xyz_m"][2] if t["xyz_m"] else float("nan")
        zm = t["xyz_mask_m"][2] if t["xyz_mask_m"] else float("nan")
        filas.append({"Z real (m)": Z, "Z SGBM (m)": zs, "error SGBM (mm)": 1000 * (zs - Z),
                      "Z máscaras (m)": zm, "error máscaras (mm)": 1000 * (zm - Z),
                      "tiempo (ms)": res["timing_ms"]["total"]})
        print(f"B={calib.baseline * 100:.0f} cm  Z real {Z:.2f} m -> SGBM {zs:.4f} m ({1000 * (zs - Z):+.1f} mm) | "
              f"máscaras {zm:.4f} m ({1000 * (zm - Z):+.1f} mm)  ({res['timing_ms']['total']:.0f} ms)")
    return filas

---
# Demostraciones
Estas celdas solo corren con `DEMO = True` (cuando abrís este notebook directamente).

### Prueba sintética
No necesita cámaras ni modelo entrenado.

In [ ]:
if DEMO:
    prueba_calibracion_sintetica()
    for B in (0.10, 0.22):
        prueba_profundidad_sintetica(calibracion_generica(baseline_m=B))

### Detectar tomates en un par real
1. Calibración: usa `stereo_calib.yml` del proyecto (generado desde la web) y, si no existe, la genérica de la C270 con 10 cm de línea base. Cambiá `BASELINE_GENERICA_M` si tus cámaras están a otra distancia.
2. Imágenes: en Colab se piden con un diálogo de subida (primero la **izquierda**, después la **derecha**). En la PC, completá `RUTA_IZQ` y `RUTA_DER`.

In [ ]:
RUTA_IZQ = None  # ej: "capturas/par_L.jpg" (solo en la PC)
RUTA_DER = None
BASELINE_GENERICA_M = 0.10
CONFIANZA = 0.35

if DEMO:
    try:
        calib = cargar_calibracion()
        print("Calibración:", calib.summary())
    except FileNotFoundError:
        calib = calibracion_generica(baseline_m=BASELINE_GENERICA_M)
        print("No hay calibración guardada; uso:", calib.created)

    if EN_COLAB and RUTA_IZQ is None:
        from google.colab import files
        print("Elegí la imagen IZQUIERDA")
        up = files.upload()
        RUTA_IZQ = next(iter(up))
        print("Elegí la imagen DERECHA")
        up = files.upload()
        RUTA_DER = next(iter(up))

    if RUTA_IZQ and RUTA_DER:
        detector = TomatoDetector()
        res = procesar_par(leer_imagen(RUTA_IZQ), leer_imagen(RUTA_DER), calib, detector, conf=CONFIANZA)
        mostrar_resultado(res)
    else:
        print("Completá RUTA_IZQ y RUTA_DER para procesar un par real.")

### Medir el error de Z
Armá un CSV `left,right,z_true_m` con pares capturados a distancias medidas con cinta y completá `CSV_MEDICIONES`.

In [ ]:
CSV_MEDICIONES = None  # ej: "capturas/mediciones.csv"

if DEMO:
    if CSV_MEDICIONES:
        evaluar_profundidad(CSV_MEDICIONES, calib, TomatoDetector())
    else:
        print("Completá CSV_MEDICIONES para evaluar la profundidad.")